# Text knowledge graph

Build the maintenance property graph from entities and structured fields.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/config.py").exists())
sys.path.insert(0, str(ROOT))
from src.data import write_json

pd.set_option("display.max_columns", 20)

from src.knowledge_graph import build_text_edges, build_text_nodes, graph_statistics

records = pd.read_csv(ROOT / "results/text/record_audit.csv", keep_default_na=False)
entities = pd.read_csv(ROOT / "results/text/entities.csv", keep_default_na=False)
out = ROOT / "results/kg/text"
out.mkdir(parents=True, exist_ok=True)
edges = build_text_edges(records, entities)
nodes = build_text_nodes(records, edges)
assert nodes["node_id"].is_unique
assert edges[["evidence", "provenance"]].notna().all().all()
assert edges[["evidence", "provenance"]].ne("").all().all()
nodes.to_csv(out / "nodes.csv", index=False)
edges.to_csv(out / "edges.csv", index=False)
display(edges.head(8))

,source,relation,target,row_id,source_field,evidence,source_dataset,provenance,certainty,span_start,span_end
0,work_order:20346569,ABOUT_EQUIPMENT,equipment:M-067,0,Equipment_ID,M-067,maintenance,structured field,observed field; not independently verified,NaN,NaN
1,work_order:20346569,HAS_MAINTENANCE_TYPE,maintenance_type:CM02,0,Maintenance_activity_type,CM02,maintenance,structured field,observed field; not independently verified,NaN,NaN
2,work_order:20381028,ABOUT_EQUIPMENT,equipment:P-573,1,Equipment_ID,P-573,maintenance,structured field,observed field; not independently verified,NaN,NaN
3,work_order:20381028,HAS_ORDER_TYPE,order_type:PM,1,OrderType,PM,maintenance,structured field,observed field; not independently verified,NaN,NaN
4,work_order:20381028,HAS_MAINTENANCE_TYPE,maintenance_type:PM01,1,Maintenance_activity_type,PM01,maintenance,structured field,observed field; not independently verified,NaN,NaN
5,work_order:20232132,ABOUT_EQUIPMENT,equipment:SC-045,2,Equipment_ID,SC-045,maintenance,structured field,observed field; not independently verified,NaN,NaN
6,work_order:20232132,HAS_ORDER_TYPE,order_type:PM,2,OrderType,PM,maintenance,structured field,observed field; not independently verified,NaN,NaN
7,work_order:20232132,HAS_MAINTENANCE_TYPE,maintenance_type:PM01,2,Maintenance_activity_type,PM01,maintenance,structured field,observed field; not independently verified,NaN,NaN


## Structure

Work orders retain all source rows, including conflicting structured fields; edge provenance preserves the disagreement. Connected components here describe shared vocabulary and field values. Generic nodes such as inspect, replace or CM can create a giant component without physical industrial connectivity.

In [2]:
summary, degree = graph_statistics(nodes, edges)
node_counts = nodes["node_type"].value_counts().rename_axis("node_type").reset_index(name="nodes")
relation_counts = edges["relation"].value_counts().rename_axis("relation").reset_index(name="edges")
degree.to_csv(out / "degree.csv", index=False)
degree.groupby("node_type")["degree"].describe().to_csv(out / "degree_by_type.csv")
node_counts.to_csv(out / "node_counts.csv", index=False)
relation_counts.to_csv(out / "relation_counts.csv", index=False)
degree.head(20).to_csv(out / "hubs.csv", index=False)
write_json(out / "summary.json", summary)
display(node_counts)
display(relation_counts)
display(degree.head(10))
display(pd.DataFrame([summary]))

,node_type,nodes
0,work_order,48613
1,equipment,9400
2,asset_mention,168
3,maintenance_type,74
4,component,18
5,maintenance_action,13
6,equipment_type,9
7,failure_mode,8
8,order_type,8


,relation,edges
0,HAS_MAINTENANCE_ACTION,167336
1,INVOLVES_COMPONENT,143530
2,HAS_ORDER_TYPE,47892
3,HAS_MAINTENANCE_TYPE,47873
4,MENTIONS_ASSET_TOKEN,47662
5,ABOUT_EQUIPMENT,47562
6,MENTIONS_EQUIPMENT_TYPE,34115
7,REPORTS_FAILURE_MODE,18046


,node_id,degree,node_type,label
9615,maintenance_action:test,33314,maintenance_action,test
9607,maintenance_action:inspect,30851,maintenance_action,inspect
9613,maintenance_action:replace,30846,maintenance_action,replace
9690,order_type:CM,25701,order_type,CM
9695,order_type:PM,19760,order_type,PM
181,component:pump,13676,component,pump
168,component:bearing,10842,component,bearing
9606,maintenance_action:clean,10544,maintenance_action,clean
182,component:seal,9930,component,seal
9628,maintenance_type:CM01,8470,maintenance_type,CM01


,nodes,edges,connected_components,largest_component,median_degree,max_degree
0,58311,554016,1,58311,9.0,33314
